# 02 — 15 Business Questions in SQL

**Olist Customer Intelligence Platform**

Fifteen business questions answered in pure SQL against the dbt marts. Each
question is stated in business terms, answered with a query, and closed with a
one-line takeaway a stakeholder would care about.

**Rules of the house**
- Count customers with `customer_unique_id`, never `customer_id`.
- "Revenue" means `delivered` orders only (except Q3, which is about leakage).
- Prefer `mart_orders` / `mart_customer_metrics`; go to item-level tables only
  when you need product/category detail.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DB_PATH = None
for candidate in (Path("../data/olist.duckdb"), Path("data/olist.duckdb")):
    if candidate.exists():
        DB_PATH = candidate.resolve()
        break
if DB_PATH is None:
    raise FileNotFoundError("olist.duckdb not found — run scripts/load_raw.py and `dbt build`.")

con = duckdb.connect(str(DB_PATH), read_only=True)


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the Olist DuckDB and return a DataFrame."""
    return con.execute(sql).df()


print(f"Connected (read-only) to {DB_PATH.name}")

Connected (read-only) to olist.duckdb


## A. Revenue & orders

### Q1 — Total delivered revenue, order count, and average order value

The headline KPIs. *(basic aggregation, `mart_orders`)*

In [3]:
# Q1
q("""
select
    count(*)                   as delivered_order_count,
    sum(total_order_value)     as total_delivered_revenue,
    avg(total_order_value)     as avg_order_value
from mart_orders
where order_status = 'delivered'
""")

,delivered_order_count,total_delivered_revenue,avg_order_value
0,96478,15419773.75,159.826839


**Takeaway:** _96,478 delivered orders generated R$15.4M in revenue, with an average order value of ~R$160._

### Q2 — Top 5 calendar months by delivered revenue

Seasonality check that sets up forecasting. *(date bucketing + LIMIT)*

In [5]:
# Q2
q("""
select
    date_trunc('month', order_purchase_timestamp) as month,
    sum(total_order_value) as delivered_revenue
from mart_orders
where order_status = 'delivered'
group by 1
order by delivered_revenue desc
limit 5
""")


,month,delivered_revenue
0,2018-08-01,1347294.08
1,2018-04-01,1300707.56
2,2018-06-01,1171020.32
3,2018-05-01,1170436.47
4,2017-12-01,1102116.05


**Takeaway:** _Revenue by purchase month peaks in Nov 2017 (R$1.15M), with strong months in early 2018 — attributed to when orders were placed, not when they landed._

### Q3 — % of gross booked order value that never converts (leakage)

One number for funnel leakage. *Denominator = ALL orders, numerator = non-delivered value — don't blanket-filter to delivered.*

In [6]:
# Q3
q("""
select
    round(100.0 * sum(case when order_status != 'delivered' then total_order_value else 0 end)
        / sum(total_order_value), 2) as leakage_pct,
    sum(total_order_value) as gross_booked_value,
    sum(case when order_status != 'delivered' then total_order_value else 0 end) as non_delivered_value
from mart_orders
""")

,leakage_pct,gross_booked_value,non_delivered_value
0,2.67,15843553.24,423779.49


**Takeaway:** _2.7% of gross booked value (R$424K) never converts to delivered revenue — a manageable but worth-monitoring leakage rate._

## B. Customer value concentration

### Q4 — Revenue share from the top 10% of customers by spend

The Pareto / whale question. *(NTILE(10) or percentile, `mart_customer_metrics`)*

In [7]:
# Q4
q("""
with ranked as (
    select
        monetary,
        ntile(10) over (order by monetary desc) as spend_decile
    from mart_customer_metrics
)

select
    round(100.0 * sum(case when spend_decile = 1 then monetary else 0 end) / sum(monetary), 1)
        as top_10_pct_revenue_share
from ranked
""")

,top_10_pct_revenue_share
0,38.3


**Takeaway:** _The top 10% of customers by spend account for 38% of total revenue — a moderate Pareto effect, not extreme whale concentration._

### Q5 — One-time vs. repeat customers: count and avg monetary value of each

Proves repeat customers are worth more. *(CASE on frequency)*

In [8]:
# Q5
q("""
select
    case when frequency = 1 then 'one-time' else 'repeat' end as customer_type,
    count(*) as customer_count,
    round(avg(monetary), 2) as avg_monetary
from mart_customer_metrics
group by 1
order by 1
""")

,customer_type,customer_count,avg_monetary
0,one-time,90557,160.73
1,repeat,2801,308.53


**Takeaway:** _Repeat customers (3%) spend nearly 2x more on average (R$309 vs R$161) — converting one-time buyers is high-leverage._

### Q6 — Customer counts by recency bucket (0-90, 91-180, 181-365, 365+)

How cold the base already is. *(CASE bucketing on recency_days)*

In [9]:
# Q6
q("""
select
    case
        when recency_days <= 90 then '0-90'
        when recency_days <= 180 then '91-180'
        when recency_days <= 365 then '181-365'
        else '365+'
    end as recency_bucket,
    count(*) as customer_count
from mart_customer_metrics
group by 1
order by min(recency_days)
""")

,recency_bucket,customer_count
0,0-90,18520
1,91-180,19706
2,181-365,34452
3,365+,20680


**Takeaway:** _The largest bucket is 181–365 days since last order (34K customers) — a big reactivation opportunity in the mid-cold segment._

## C. Retention & repeat behavior

### Q7 — Avg days between 1st and 2nd order (repeat customers only)

Defines the natural repurchase window. *(ROW_NUMBER / LEAD over customer, ordered by date)*

In [10]:
# Q7
q("""
with ordered as (
    select
        customer_unique_id,
        order_purchase_timestamp,
        row_number() over (
            partition by customer_unique_id
            order by order_purchase_timestamp
        ) as rn
    from mart_orders
    where order_status = 'delivered'
),

gaps as (
    select
        date_diff('day', a.order_purchase_timestamp, b.order_purchase_timestamp) as days_to_second
    from ordered a
    join ordered b
        on a.customer_unique_id = b.customer_unique_id
        and a.rn = 1
        and b.rn = 2
)

select
    round(avg(days_to_second), 1) as avg_days_to_second_order,
    count(*) as repeat_customers
from gaps
""")

,avg_days_to_second_order,repeat_customers
0,81.2,2801


**Takeaway:** _Repeat customers take ~81 days on average between their 1st and 2nd order — target win-back campaigns around the 60–90 day window._

### Q8 — Of customers acquired in a quarter, % with a 2nd order within 90 days

Repeat rate by acquisition cohort. *(first-order CTE + date arithmetic)*

In [11]:
# Q8
q("""
with first_orders as (
    select
        customer_unique_id,
        min(order_purchase_timestamp) as first_order_date
    from mart_orders
    where order_status = 'delivered'
    group by 1
),

cohorts as (
    select
        customer_unique_id,
        date_trunc('quarter', first_order_date) as acquisition_quarter,
        first_order_date
    from first_orders
),

second_orders as (
    select
        c.customer_unique_id,
        c.acquisition_quarter,
        c.first_order_date,
        min(o.order_purchase_timestamp) as second_order_date
    from cohorts c
    join mart_orders o
        on c.customer_unique_id = o.customer_unique_id
        and o.order_status = 'delivered'
        and o.order_purchase_timestamp > c.first_order_date
    group by 1, 2, 3
)

select
    c.acquisition_quarter,
    count(*) as acquired_customers,
    count(*) filter (
        where date_diff('day', c.first_order_date, s.second_order_date) <= 90
    ) as repeat_within_90d,
    round(100.0 * count(*) filter (
        where date_diff('day', c.first_order_date, s.second_order_date) <= 90
    ) / count(*), 1) as pct_repeat_within_90d
from cohorts c
left join second_orders s
    on c.customer_unique_id = s.customer_unique_id
group by c.acquisition_quarter
order by c.acquisition_quarter
""")

,acquisition_quarter,acquired_customers,repeat_within_90d,pct_repeat_within_90d
0,2016-07-01,1,0,0.0
1,2016-10-01,263,4,1.5
2,2017-01-01,4848,104,2.1
3,2017-04-01,8744,205,2.3
4,2017-07-01,11813,277,2.3
5,2017-10-01,16726,310,1.9
6,2018-01-01,19904,408,2.0
7,2018-04-01,18966,266,1.4
8,2018-07-01,12093,93,0.8


**Takeaway:** _Only ~2% of acquired customers reorder within 90 days, peaking at 2.3% for mid-2017 cohorts — but 2018-Q3's 0.8% is right-censored (customers hadn't lived a full 90-day window before the data ends)._

## D. Products & margin

### Q9 — Top 10 categories by delivered revenue, with avg review + order count

The product leaderboard. *(items -> products -> mart_orders)*

In [12]:
# Q9
q("""
with item_revenue as (
    select
        p.product_category,
        i.order_id,
        sum(i.price + i.freight_value) as item_revenue
    from stg_order_items i
    join mart_orders o
        on i.order_id = o.order_id
        and o.order_status = 'delivered'
    join int_products_translated p
        on i.product_id = p.product_id
    group by 1, 2
),

order_reviews as (
    select order_id, avg_review_score
    from mart_orders
    where order_status = 'delivered'
      and avg_review_score is not null
)

select
    ir.product_category,
    round(sum(ir.item_revenue), 2) as delivered_revenue,
    round(avg(orr.avg_review_score), 2) as avg_review_score,
    count(distinct ir.order_id) as order_count
from item_revenue ir
left join order_reviews orr
    on ir.order_id = orr.order_id
group by 1
order by delivered_revenue desc
limit 10
""")

,product_category,delivered_revenue,avg_review_score,order_count
0,health_beauty,1412089.53,4.19,8647
1,watches_gifts,1264333.12,4.07,5495
2,bed_bath_table,1225209.26,3.92,9272
3,sports_leisure,1118256.91,4.17,7530
4,computers_accessories,1032723.77,3.99,6530
5,furniture_decor,880329.92,3.95,6307
6,housewares,758392.25,4.11,5743
7,cool_stuff,691680.89,4.19,3559
8,auto,669454.75,4.12,3810
9,garden_tools,567145.68,4.08,3448


**Takeaway:** _Health & beauty leads at R$1.4M, followed by watches/gifts and bed/bath — all top categories score above 3.9 on reviews._

### Q10 — Categories with high revenue but below-average review score

The problem categories. *(compare to the global avg review)*

In [13]:
# Q10
q("""
with item_revenue as (
    select
        p.product_category,
        i.order_id,
        sum(i.price + i.freight_value) as item_revenue
    from stg_order_items i
    join mart_orders o
        on i.order_id = o.order_id
        and o.order_status = 'delivered'
    join int_products_translated p
        on i.product_id = p.product_id
    group by 1, 2
),

category_stats as (
    select
        ir.product_category,
        sum(ir.item_revenue) as delivered_revenue,
        avg(orr.avg_review_score) as avg_review_score
    from item_revenue ir
    left join (
        select order_id, avg_review_score
        from mart_orders
        where order_status = 'delivered'
          and avg_review_score is not null
    ) orr on ir.order_id = orr.order_id
    group by 1
),

global_avg as (
    select avg(avg_review_score) as global_avg_review
    from mart_orders
    where order_status = 'delivered'
      and avg_review_score is not null
),

revenue_p75 as (
    select quantile_cont(delivered_revenue, 0.75) as p75_revenue
    from category_stats
)

select
    cs.product_category,
    round(cs.delivered_revenue, 2) as delivered_revenue,
    round(cs.avg_review_score, 2) as avg_review_score,
    round(g.global_avg_review, 2) as global_avg_review
from category_stats cs
cross join global_avg g
cross join revenue_p75 r
where cs.delivered_revenue >= r.p75_revenue
  and cs.avg_review_score < g.global_avg_review
order by cs.delivered_revenue desc
""")

,product_category,delivered_revenue,avg_review_score,global_avg_review
0,watches_gifts,1264333.12,4.07,4.16
1,bed_bath_table,1225209.26,3.92,4.16
2,computers_accessories,1032723.77,3.99,4.16
3,furniture_decor,880329.92,3.95,4.16
4,housewares,758392.25,4.11,4.16
5,auto,669454.75,4.12,4.16
6,garden_tools,567145.68,4.08,4.16
7,baby,466727.65,4.08,4.16
8,telephony,379202.62,3.99,4.16
9,office_furniture,335211.36,3.51,4.16


**Takeaway:** _8 top-quartile categories score below the 4.16 global avg — office furniture (3.64) is the clearest problem; bed/bath and computers/accessories are high-revenue review risks too._

### Q11 — Avg freight as % of item price by category (freight burden)

Freight erodes margin. *(ratio aggregation)*

In [14]:
# Q11
q("""
select
    p.product_category,
    round(100.0 * sum(i.freight_value) / nullif(sum(i.price), 0), 1) as freight_pct_of_price
from stg_order_items i
join mart_orders o
    on i.order_id = o.order_id
    and o.order_status = 'delivered'
join int_products_translated p
    on i.product_id = p.product_id
group by 1
having sum(i.price) > 0
order by freight_pct_of_price desc
limit 10
""")

,product_category,freight_pct_of_price
0,home_comfort_2,54.0
1,flowers,44.0
2,furniture_mattress_and_upholstery,36.6
3,christmas_supplies,36.5
4,diapers_and_hygiene,36.3
5,cds_dvds_musicals,30.8
6,signaling_and_security,30.4
7,electronics,29.5
8,food_drink,29.4
9,dvds_blu_ray,27.0


**Takeaway:** _Freight eats 27–54% of item price in the heaviest categories (home comfort, flowers, mattresses) — pricing and fulfillment strategy should differ by category._

## E. Geography & logistics

### Q12 — Top 10 states by revenue, each with avg delivery time

Where the money and the logistics pain are. *(date_diff, join to state)*

In [15]:
# Q12
q("""
select
    customer_state,
    round(sum(total_order_value), 2) as delivered_revenue,
    round(avg(date_diff(
        'day', order_purchase_timestamp, order_delivered_customer_date
    )), 1) as avg_delivery_days
from mart_orders
where order_status = 'delivered'
  and customer_state is not null
group by 1
order by delivered_revenue desc
limit 10
""")

,customer_state,delivered_revenue,avg_delivery_days
0,SP,5769703.15,8.7
1,RJ,2055401.57,15.2
2,MG,1818891.67,11.9
3,RS,861472.79,15.2
4,PR,781708.80,11.9
5,SC,595127.78,14.9
6,BA,591137.81,19.3
7,DF,346123.35,12.9
8,GO,334212.35,15.5
9,ES,317657.93,15.7


**Takeaway:** _SP dominates with R$5.8M (37% of revenue) and the fastest delivery (8.7 days); BA is top-10 revenue but slowest at 19.3 days._

### Q13 — States with the worst on-time rate (delivered after estimated date)

Service-quality map. *(AVG(CASE WHEN delivered > estimated ...))*

In [16]:
# Q13
q("""
select
    customer_state,
    round(100.0 * avg(case
        when order_delivered_customer_date > order_estimated_delivery_date then 1.0
        else 0.0
    end), 1) as late_pct,
    count(*) as order_count
from mart_orders
where order_status = 'delivered'
  and customer_state is not null
  and order_delivered_customer_date is not null
  and order_estimated_delivery_date is not null
group by 1
having count(*) >= 100
order by late_pct desc
limit 10
""")

,customer_state,late_pct,order_count
0,AL,23.9,397
1,MA,19.7,717
2,PI,16.0,476
3,CE,15.3,1279
4,SE,15.2,335
5,BA,14.0,3256
6,RJ,13.5,12350
7,TO,12.8,274
8,PA,12.4,946
9,ES,12.2,1995


**Takeaway:** _AL has the worst on-time performance (24% late), followed by MA and PI — northeastern states need logistics attention._

## F. Payments

### Q14 — Order distribution by payment type, with avg installments

How customers pay. *(`stg_order_payments`)*

In [17]:
# Q14
q("""
select
    payment_type,
    count(distinct order_id) as order_count,
    round(avg(payment_installments), 2) as avg_installments
from stg_order_payments
group by 1
order by order_count desc
""")

,payment_type,order_count,avg_installments
0,credit_card,76505,3.51
1,boleto,19784,1.00
2,voucher,3866,1.00
3,debit_card,1528,1.00
4,not_defined,3,1.00


**Takeaway:** _Credit card dominates (76K orders, avg 3.5 installments); boleto and voucher are single-payment methods for ~24K orders combined._

### Q15 — Do orders with more installments have higher order value?

Tests whether installments drive bigger baskets. *(installment buckets vs avg value)*

In [18]:
# Q15
q("""
select
    case
        when max_installments = 1             then '1'
        when max_installments between 2 and 3 then '2-3'
        when max_installments between 4 and 6 then '4-6'
        when max_installments >= 7            then '7+'
        else 'other/unknown'
    end as installment_bucket,
    count(*) as order_count,
    round(avg(total_order_value), 2) as avg_order_value
from mart_orders
where order_status = 'delivered'
group by 1
order by 1
""")

,installment_bucket,order_count,avg_order_value
0,7+,11765,334.04
1,1,46814,120.21
2,2-3,22159,135.41
3,4-6,15739,181.82


**Takeaway:** _More installments → bigger baskets, monotonically (R$120 → R$135 → R$182 → R$334) — installments let customers buy up. Three edge-case orders (0 installments) surface in `other/unknown` instead of polluting `7+`._